In [1]:
import sys
print(sys.executable)
print(sys.version)

/usr/local/bin/python3
3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:
!pip -q install transformers datasets torch scikit-learn

In [3]:
import torch
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from sklearn.metrics import classification_report
print("All good ✅", torch.__version__)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All good ✅ 2.9.1


In [4]:
from datasets import load_dataset

dataset = load_dataset("amazon_polarity")

train_dataset = dataset["train"].select(range(500))
test_dataset  = dataset["test"].select(range(200))

print(train_dataset[0])
print("\nTrain size:", len(train_dataset))
print("Test size:", len(test_dataset))

Generating test split: 100%|██████████| 400000/400000 [00:00<00:00, 2771252.16 examples/s]


{'label': 1, 'title': 'Stuning even for the non-gamer', 'content': 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'}

Train size: 500
Test size: 200


In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 128

def tokenize_function(batch):
    return tokenizer(
        batch["content"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
train_dataset.set_format(type="torch", columns=cols)
test_dataset.set_format(type="torch", columns=cols)

print("Tokenization complete.")

Map: 100%|██████████| 200/200 [00:00<00:00, 20399.82 examples/s]

Tokenization complete.


In [6]:
import torch.nn as nn
from transformers import BertModel

class CustomBERTClassifier(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", num_labels=2):
        super(CustomBERTClassifier, self).__init__()
        
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size
        
        # Classification head
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.classifier(x)

        return logits

In [9]:
import torch
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomBERTClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

print("Model ready on:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2217.52it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready on: cpu


In [10]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready.")

DataLoaders ready.


In [11]:
from tqdm.auto import tqdm

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(1, len(dataloader))

In [12]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report
import numpy as np

def evaluate_with_metrics(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask, token_type_ids)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary"
    )

    print(f"\nEvaluation Loss: {avg_loss:.4f}")
    print(f"Evaluation Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds))

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [13]:
import pandas as pd

NUM_EPOCHS = 3
history = []

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Train Loss: {train_loss:.4f}")

    metrics = evaluate_with_metrics(model, test_loader, criterion, device)
    metrics["epoch"] = epoch + 1
    metrics["train_loss"] = train_loss
    metrics["model"] = "BASE"
    history.append(metrics)

base_results = pd.DataFrame(history)
base_results


===== Epoch 1/3 =====


Training: 100%|██████████| 32/32 [00:48<00:00,  1.53s/it]


Train Loss: 0.6336


Evaluating: 100%|██████████| 13/13 [00:04<00:00,  2.72it/s]



Evaluation Loss: 0.4545
Evaluation Accuracy: 0.8050
Precision: 0.8070
Recall: 0.8440
F1 Score: 0.8251

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.76      0.78        91
           1       0.81      0.84      0.83       109

    accuracy                           0.81       200
   macro avg       0.80      0.80      0.80       200
weighted avg       0.80      0.81      0.80       200


===== Epoch 2/3 =====


Training: 100%|██████████| 32/32 [00:47<00:00,  1.48s/it]


... (4 outputs left out)

Train Loss: 0.1469


Evaluating: 100%|██████████| 13/13 [00:04<00:00,  2.71it/s]



Evaluation Loss: 0.3581
Evaluation Accuracy: 0.8350
Precision: 0.9524
Recall: 0.7339
F1 Score: 0.8290

Classification Report:

              precision    recall  f1-score   support

           0       0.75      0.96      0.84        91
           1       0.95      0.73      0.83       109

    accuracy                           0.83       200
   macro avg       0.85      0.84      0.83       200
weighted avg       0.86      0.83      0.83       200



,loss,accuracy,precision,recall,f1,epoch,train_loss,model
0,0.454543,0.805,0.807018,0.844037,0.825112,1,0.633583,BASE
1,0.310948,0.885,0.883929,0.908257,0.895928,2,0.322048,BASE
2,0.358111,0.835,0.952381,0.733945,0.829016,3,0.146911,BASE


In [14]:
class CustomBERTClassifierV2(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", num_labels=2):
        super(CustomBERTClassifierV2, self).__init__()
        
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        # NEW deeper classification head
        self.fc1 = nn.Linear(hidden_size, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)

        return logits

In [15]:
BATCH_SIZE = 32
NUM_EPOCHS = 5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model_v2 = CustomBERTClassifierV2().to(device)
optimizer_v2 = AdamW(model_v2.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

history_v2 = []

for epoch in range(NUM_EPOCHS):
    print(f"\n===== V2 Epoch {epoch+1}/{NUM_EPOCHS} =====")
    
    train_loss = train_one_epoch(model_v2, train_loader, optimizer_v2, criterion, device)
    print(f"Train Loss: {train_loss:.4f}")

    metrics = evaluate_with_metrics(model_v2, test_loader, criterion, device)
    metrics["epoch"] = epoch + 1
    metrics["train_loss"] = train_loss
    metrics["model"] = "V2"
    history_v2.append(metrics)

v2_results = pd.DataFrame(history_v2)
v2_results

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2232.74it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== V2 Epoch 1/5 =====


Training: 100%|██████████| 16/16 [00:50<00:00,  3.15s/it]


Train Loss: 0.6984


Evaluating: 100%|██████████| 7/7 [00:04<00:00,  1.49it/s]



Evaluation Loss: 0.6769
Evaluation Accuracy: 0.5700
Precision: 0.7255
Recall: 0.3394
F1 Score: 0.4625

Classification Report:

              precision    recall  f1-score   support

           0       0.52      0.85      0.64        91
           1       0.73      0.34      0.46       109

    accuracy                           0.57       200
   macro avg       0.62      0.59      0.55       200
weighted avg       0.63      0.57      0.54       200


===== V2 Epoch 2/5 =====


... (13 outputs left out)

Train Loss: 0.1355


Evaluating: 100%|██████████| 7/7 [00:05<00:00,  1.37it/s]


Evaluation Loss: 0.3437
Evaluation Accuracy: 0.8700
Precision: 0.8952
Recall: 0.8624
F1 Score: 0.8785

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.88      0.86        91
           1       0.90      0.86      0.88       109

    accuracy                           0.87       200
   macro avg       0.87      0.87      0.87       200
weighted avg       0.87      0.87      0.87       200



,loss,accuracy,precision,recall,f1,epoch,train_loss,model
0,0.676854,0.570,0.725490,0.339450,0.462500,1,0.698366,V2
1,0.505631,0.785,0.811321,0.788991,0.800000,2,0.600865,V2
2,0.353013,0.840,0.888889,0.807339,0.846154,3,0.388966,V2
3,0.337332,0.865,0.836066,0.935780,0.883117,4,0.238649,V2
4,0.343656,0.870,0.895238,0.862385,0.878505,5,0.135525,V2


In [16]:
base_results.to_csv("base_model_results.csv", index=False)
v2_results.to_csv("v2_model_results.csv", index=False)